# Experiments for Model 1 (PCCILM): fairer labels, better eyes, honest evaluation

Same data as `02_train_colab_neon.ipynb` (events from **Neon**, screenshots from the **bucket**), but
instead of one training run this notebook runs three experiments. `02` is untouched and keeps
working as before.

| Experiment | Question | How |
|---|---|---|
| **1. Title cleanup (labels v4)** | Do changing window titles hide real revisits? | Labels are built twice: exact titles (v3) and cleaned titles (v4). Cleaning removes Edge's `and N more pages`, `(8)` counters, `●` unsaved markers and invisible characters |
| **2. Screen categories** | Can MobileCLIP's "eyes" tell Model 1 something the app name and title can't? | MobileCLIP compares every screenshot with descriptions like *"an online shop product page"* or *"source code"* and gives Model 1 a score per category |
| **3. Rolling evaluation** | Is Model 1 *reliably* better than the rules, or did one split get lucky? | Train on everything before a test period, test on that period, then move forward. Repeated over several consecutive periods, reported as average ± spread, plus latency |

**Before running:** the same as `02`: T4 GPU runtime, the Colab secret `DATABASE_URL`, and the latest
code pushed to GitHub (this notebook needs `offline/cloud_sources.py`, `features/titles.py` and the
updated `offline/training.py`).

**Fast if you've run `02`:** embeddings are reused from the same Drive cache, so the GPU step is
skipped for screenshots already embedded.

> Outputs show screenshots and app names. Don't share or commit this notebook with outputs.

## 0. Settings

In [ ]:
PROFILE_ID = "imeth-k7mx9q2n_b4r"   # whose data (other participants' rows are never read)
DB_SCHEMA = "drc"
BUCKET = "desktop-research-data-collection"
GCP_PROJECT = "project-dd3132f1-b1c9-49ea-bb8"

REPO_URL = "https://github.com/imethsuka/J26-IT-308.git"
REPO_BRANCH = "main"
WORKER_SUBDIR = "context-importance-system/python-worker"

FROM_DATE = None                    # e.g. "2026-09-20"; None = everything for PROFILE_ID
TO_DATE = None
CACHE_EMBEDDINGS_ON_DRIVE = True    # same cache as notebook 02, so embeddings are reused

N_FOLDS = 4                         # rolling evaluation: how many consecutive test periods
ROLLING_TEST_FRACTION = 0.5         # the newest half of labelled data is split into those periods
PROTOTYPE_K = 8
THRESHOLD = 0.5
SHOW_SCREENSHOTS = True             # example screenshots per screen category

In [ ]:
import sys
from pathlib import Path

from google.colab import auth

auth.authenticate_user()

IMAGES = Path("/content/screenshots") / PROFILE_ID      # temporary: gone when the session ends
WORK = Path("/content/work") / PROFILE_ID
if CACHE_EMBEDDINGS_ON_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/context-importance") / PROFILE_ID
else:
    CACHE = WORK
for p in (IMAGES, WORK, CACHE):
    p.mkdir(parents=True, exist_ok=True)

## 1. Code and packages

In [ ]:
import subprocess

repo = Path("/content/J26-IT-308")
if repo.exists():  # pick up anything pushed since this session started
    subprocess.run(["git", "-C", str(repo), "pull", "--quiet", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo)], check=True)
WORKER_DIR = repo / WORKER_SUBDIR
COLLECTOR_CONFIG = WORKER_DIR.parent / "csharp-collector" / "src" / "CsharpCollector" / "collector.config.json"
for needed in ("offline/cloud_sources.py", "features/titles.py"):
    assert (WORKER_DIR / needed).is_file(), f"{needed} not on GitHub yet: commit and push it first"
sys.path.insert(0, str(WORKER_DIR))
print(subprocess.run(["git", "-C", str(repo), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q lightgbm river timm open_clip_torch "sqlalchemy>=2" psycopg2-binary pyarrow
%pip install -q --no-deps git+https://github.com/apple/ml-mobileclip.git

# If pip upgraded numpy under an already-loaded numpy, later cells fail ("cannot import name
# '_slice'..."). Restart once; on the next run nothing needs upgrading.
import importlib.metadata

import numpy

if importlib.metadata.version("numpy") != numpy.__version__:
    print("numpy was upgraded: Colab is restarting the session. Wait for it to reconnect, then Run all again.")
    import os

    os.kill(os.getpid(), 9)
print("packages ready, numpy", numpy.__version__)

In [ ]:
import urllib.request

import torch

CHECKPOINT = Path("/content/mobileclip_s0.pt")
if not CHECKPOINT.exists():
    urllib.request.urlretrieve(
        "https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_s0.pt", CHECKPOINT
    )
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Data: Neon events + bucket screenshots → captures → embeddings

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import storage
from google.colab import userdata
from sqlalchemy import create_engine
from sqlalchemy.engine import make_url

from features.mobileclip_encoder import encode_texts
from features.titles import normalize_title
from models.rule_based import RuleBasedImportanceScorer
from offline.cloud_sources import clean_database_url, download_profile_screenshots, fetch_profile_tables, write_drc_sqlite
from offline.drc_replay import Exclusions, replay_drc
from offline.training import (
    SCREEN_CATEGORIES, add_labels, build_dataset, category_columns, embed_captures, rolling_evaluation,
    summarize_folds, train_model1, with_categories, without_vision,
)

pd.set_option("display.width", 180)

url = clean_database_url(userdata.get("DATABASE_URL"))
safe = make_url(url)
print(f"Neon: {safe.host} / {safe.database} as {safe.username} (password hidden)")
engine = create_engine(url, pool_pre_ping=True)
tables = fetch_profile_tables(engine, PROFILE_ID, schema=DB_SCHEMA, from_date=FROM_DATE, to_date=TO_DATE)
engine.dispose()
for name, df in tables.items():
    print(f"{name:20} {len(df):8} rows")

drc_db = write_drc_sqlite(tables, WORK / "neon_as_drc.db")
counts = download_profile_screenshots(storage.Client(project=GCP_PROJECT), BUCKET, PROFILE_ID, IMAGES)
print("screenshots:", counts)

captures, replay_stats = replay_drc(drc_db, exclusions=Exclusions.from_collector_config(COLLECTOR_CONFIG), screenshots_dir=IMAGES)
print(replay_stats.summary())
embeddings = embed_captures(captures, CACHE / "embeddings.npz", checkpoint=str(CHECKPOINT), device=DEVICE)
print(f"{sum(c.screenshot_id in embeddings for c in captures)} of {len(captures)} captures embedded")

## Experiment 1: title cleanup (labels v3 vs v4)
Both datasets use identical captures and embeddings. The only difference is whether window titles
are cleaned before deciding "is this the same page as before?", for both the label and the revisit
features.

In [ ]:
data_v3 = add_labels(build_dataset(captures, embeddings))
data_v4 = add_labels(build_dataset(captures, embeddings, title_normalizer=normalize_title), title_normalizer=normalize_title)

both = data_v3["label"].notna() & data_v4["label"].notna()
a, b = data_v3.loc[both, "label"].astype(int), data_v4.loc[both, "label"].astype(int)
print(f"labelled captures: {both.sum()}")
print(f"important:  v3 {a.mean():.1%}  ->  v4 {b.mean():.1%}")
print(f"labels changed: not important -> important {((a == 0) & (b == 1)).sum()},  important -> not important {((a == 1) & (b == 0)).sum()}")
changed = data_v3.loc[both][(a == 0) & (b == 1)]
display(changed["app"].value_counts().head(8).rename("labels that became 'important'").to_frame())

## Experiment 2: screen categories from MobileCLIP
MobileCLIP's text half turns each description below into the same kind of 512 numbers as a
screenshot. A screenshot's closeness to each description becomes a probability: *how much does this
look like a shop page / code / a chart …*. Edit `SCREEN_CATEGORIES` (in `offline/training.py`) or the
dict below to try other categories; no labels are used, so this can't leak test answers.

In [ ]:
CATEGORIES = dict(SCREEN_CATEGORIES)          # name -> description; edit here to experiment
CAT_COLS = category_columns(CATEGORIES)
text_matrix = encode_texts(list(CATEGORIES.values()), CHECKPOINT, device=DEVICE)

data_v3 = with_categories(data_v3, text_matrix, list(CATEGORIES))
data_v4 = with_categories(data_v4, text_matrix, list(CATEGORIES))

cats = pd.DataFrame([{c: f[c] for c in CAT_COLS} for f in data_v4["features"]])
data_v4["top_category"] = cats.idxmax(axis=1).str.replace("vis_cat_", "", regex=False)
has_emb = data_v4["embedding"].notna()
print("Which category MobileCLIP picks most often, per app (rows = apps):")
display(pd.crosstab(data_v4.loc[has_emb, "app"], data_v4.loc[has_emb, "top_category"])
        .loc[lambda t: t.sum(axis=1).sort_values(ascending=False).index[:10]])
labelled = data_v4[data_v4["label"].notna() & has_emb]
print("Share labelled important (v4), by top category:")
display(labelled.groupby("top_category")["label"].agg(captures="size", important="mean").sort_values("captures", ascending=False).round(2))

In [ ]:
if SHOW_SCREENSHOTS:
    from PIL import Image

    top = data_v4.loc[has_emb, "top_category"].value_counts().index[:8]
    fig, axes = plt.subplots(len(top), 3, figsize=(10, len(top) * 1.9))
    for r, cat in enumerate(top):
        col = f"vis_cat_{cat}"
        rows = data_v4[has_emb].assign(p=[f[col] for f in data_v4.loc[has_emb, "features"]]).nlargest(3, "p")
        for c, row in enumerate(rows.itertuples()):
            with Image.open(row.image_path) as im:
                axes[r, c].imshow(im.convert("RGB").resize((320, 180)))
            axes[r, c].set_axis_off()
            axes[r, c].set_title(f"{cat} ({row.p:.2f})" if c == 0 else "", fontsize=8, loc="left")
    plt.tight_layout()
    plt.show()

## Experiment 3: rolling evaluation
The newest half of the labelled data is cut into `N_FOLDS` consecutive periods. For each period,
every model trains only on data from **before** it (with the 48-hour gap) and is tested on it.
Each configuration is run under both label versions.

**Read the first four metrics first. They're fair between the rules and Model 1:**
- **roc_auc / avg_precision**: how well the scores *order* screens from most to least important,
  whatever threshold is used. 0.5 / the share of important screens = random; 1.0 = perfect.
- **recall_at_keep_30 / _50**: if only 30% (or 50%) of screens may be stored, what share of the
  important ones is kept? This is your storage-vs-memory claim directly.
- **f1 / keep_rate** at threshold 0.5. Careful: the rules and Model 1 are calibrated differently, so at
  0.5 the rules keep more screens, and F1 tends to reward keeping more. A local preview on 8 to 19 Oct
  showed exactly this: the rules won on F1, but Model 1 ranked better (AUC 0.79–0.85 vs 0.72–0.76).
- **mean** = typical result, **std** = variation between periods (smaller = more reliable),
  **folds_won** = periods in which that configuration had the best average precision
- **latency_mean_ms** = one keep/discard decision by the model (Colab CPU; MobileCLIP is timed below)

Compare configurations **within** one label version. v3 and v4 score against different answers, so
a higher F1 under v4 doesn't by itself mean a better model.

In [ ]:
configs = {
    "Rules (live today)": lambda tr, te: (RuleBasedImportanceScorer(), te),
    "LightGBM": lambda tr, te: (train_model1(tr)[0], te),
    "LightGBM without eyes": lambda tr, te: (train_model1(without_vision(tr))[0], without_vision(te)),
    "LightGBM + screen categories": lambda tr, te: (train_model1(tr, extra_feature_columns=CAT_COLS)[0], te),
}
fold_results = {}
for version, data in (("v3 (exact titles)", data_v3), ("v4 (cleaned titles)", data_v4)):
    fold_results[version] = rolling_evaluation(
        data, configs, n_folds=N_FOLDS, test_fraction=ROLLING_TEST_FRACTION,
        prototype_k=PROTOTYPE_K, threshold=THRESHOLD,
    )
    print(f"\n=== labels {version}")
    display(summarize_folds(fold_results[version]))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for col, (version, res) in enumerate(fold_results.items()):
    for row, metric in enumerate(("avg_precision", "recall_at_keep_30")):
        ax = axes[row, col]
        for name, g in res.groupby("name", sort=False):
            ax.plot(g["fold"], g[metric], marker="o", label=name)
        ax.set_title(f"{metric}, labels {version}", fontsize=10)
        ax.set_xlabel("test period (later ->)")
axes[0, 1].legend(fontsize=8)
plt.tight_layout()
plt.show()
display(pd.concat(fold_results, names=["labels"])[[
    "fold", "test_from", "test_to", "train_rows", "test_rows", "name",
    "roc_auc", "avg_precision", "recall_at_keep_30", "recall_at_keep_50", "f1", "keep_rate",
]].round(3))

### What Model 1 relies on, v3 vs v4
Trained on all labelled data, with categories. If title cleanup removed a labelling artefact,
`ctx_title_token_count` should drop in importance under v4.

In [ ]:
from offline.training import learn_prototypes, with_prototypes

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (version, data) in zip(axes, (("v3", data_v3), ("v4", data_v4))):
    train_all = data[data["label"].notna()]
    proto = learn_prototypes(train_all, k=PROTOTYPE_K)
    train_all = with_prototypes(train_all, proto.centroids if proto is not None else None)
    model, _ = train_model1(train_all, extra_feature_columns=CAT_COLS)
    imp = pd.Series(model.feature_importances("gain"))
    imp = (100 * imp / imp.sum()).head(15)[::-1]
    imp.plot.barh(ax=ax, title=f"labels {version}: share of gain (%)")
    ranking = list(model.feature_importances("gain"))
    print(f"{version}: ctx_title_token_count is #{ranking.index('ctx_title_token_count') + 1} of {len(ranking)} inputs")
plt.tight_layout()
plt.show()

### MobileCLIP latency
The model's decision time is in the table above (`latency_mean_ms`). The eyes cost more, so this
times one screenshot embedding on Colab's GPU and CPU. Your own PC is the number that counts: the
local benchmark on the i5-11400H measured about 107 ms (median) on CPU
(`python -m eval.measure_encoder`).

In [ ]:
import time

from PIL import Image

from features.mobileclip_encoder import load_image_encoder

sample = [c.image_path for c in captures[:30]]
for device in dict.fromkeys([DEVICE, "cpu"]):
    encoder = load_image_encoder(str(CHECKPOINT), device=device)
    encoder.embed_frame(Image.open(sample[0]))  # warm-up
    times = []
    for path in sample:
        with Image.open(path) as im:
            start = time.perf_counter()
            encoder.embed_frame(im)
            times.append((time.perf_counter() - start) * 1000)
    print(f"MobileCLIP-S0 on Colab {device}: median {np.median(times):.1f} ms, p95 {np.percentile(times, 95):.1f} ms per screenshot")

## Save the results
Only the numbers (no screenshots, titles or apps) are written: one CSV per label version with every
fold's metrics. Paste them into your report or compare with later runs.

In [ ]:
from google.colab import files

stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d-%H%M")
for version, res in fold_results.items():
    tag = version.split()[0]
    out = WORK / f"rolling-{tag}-{stamp}.csv"
    res.to_csv(out, index=False)
    files.download(str(out))